# Lecture 3 — Connecting to APIs

This notebook demonstrates how to retrieve data from web APIs using Python's `requests`
library. APIs (Application Programming Interfaces) provide a standardised way to access
data from online services.

We use free, public APIs that do not require authentication to keep things simple in this notebook.

**Credits:**

Created by Sippo Rossi for the course Python Programming for Business Intelligence at Hanken.

## 1. What is an API?

An API is a mechanism that allows programs to communicate with each other. In practice,
you send a request to a URL and receive structured data (usually JSON) in return.

Many organisations offer APIs: X (Twitter), Yahoo Finance, government open data portals, etc.

**Important:** Many APIs require an API key, a unique identifier linked to your account.
API keys should be treated like passwords: never share them and never upload them to GitHub.

## 2. Making requests with `requests`

In [1]:
import requests
import pandas as pd

### 2.1 A simple GET request

We start with a free API that returns exchange rates. A GET request retrieves data from
the specified URL.

In [2]:
# Fetch latest EUR exchange rates from a free API
url = "https://open.er-api.com/v6/latest/EUR"
response = requests.get(url)

print("Status code:", response.status_code)
print("A status code of 200 means the request was successful")

Status code: 200
A status code of 200 means the request was successful


### 2.2 Working with JSON responses

Most APIs return data in JSON format, which maps naturally to Python dictionaries.

In [3]:
# Parse the JSON response
data = response.json()

# See what keys are available
print("Keys:", list(data.keys()))

Keys: ['result', 'provider', 'documentation', 'terms_of_use', 'time_last_update_unix', 'time_last_update_utc', 'time_next_update_unix', 'time_next_update_utc', 'time_eol_unix', 'base_code', 'rates']


In [4]:
# Access specific exchange rates
rates = data["rates"]
print(f"1 EUR = {rates['USD']} USD")
print(f"1 EUR = {rates['GBP']} GBP")
print(f"1 EUR = {rates['SEK']} SEK")

1 EUR = 1.154564 USD
1 EUR = 0.865088 GBP
1 EUR = 10.865584 SEK


## 3. Passing parameters to an API

Many APIs accept parameters that control what data is returned. These can be passed
as a dictionary.

In [5]:
# The Open-Meteo weather API - get a 7-day forecast for Helsinki
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 60.17,
    "longitude": 24.94,
    "daily": "temperature_2m_max,temperature_2m_min",
    "timezone": "Europe/Helsinki"
}

response = requests.get(url, params=params)
weather = response.json()

print("Status code:", response.status_code)

Status code: 200


In [6]:
# Inspect the daily forecast data
daily = weather["daily"]

for i in range(len(daily["time"])):
    date = daily["time"][i]
    t_max = daily["temperature_2m_max"][i]
    t_min = daily["temperature_2m_min"][i]
    print(f"{date}: {t_min}°C - {t_max}°C")

2026-03-27: 2.5°C - 6.7°C
2026-03-28: 2.0°C - 5.5°C
2026-03-29: 1.9°C - 5.6°C
2026-03-30: -0.7°C - 2.8°C
2026-03-31: 0.2°C - 8.5°C
2026-04-01: 1.7°C - 11.5°C
2026-04-02: -0.4°C - 9.6°C


## 4. Converting API data to a DataFrame

Since API responses are typically dictionaries or lists of dictionaries, they can be
converted to DataFrames conveniently.

In [7]:
# Convert the weather data to a DataFrame
df_weather = pd.DataFrame({
    "date": daily["time"],
    "temp_max": daily["temperature_2m_max"],
    "temp_min": daily["temperature_2m_min"]
})

df_weather

,date,temp_max,temp_min
0,2026-03-27,6.7,2.5
1,2026-03-28,5.5,2.0
2,2026-03-29,5.6,1.9
3,2026-03-30,2.8,-0.7
4,2026-03-31,8.5,0.2
5,2026-04-01,11.5,1.7
6,2026-04-02,9.6,-0.4


In [8]:
# Convert the exchange rates to a DataFrame
df_rates = pd.DataFrame(
    list(rates.items()),
    columns=["currency", "rate"]
)

df_rates.head(10)

,currency,rate
0,EUR,1.000000
1,AED,4.240116
2,AFN,72.963602
3,ALL,96.076877
4,AMD,436.284706
5,ANG,2.066660
6,AOA,1095.471035
7,ARS,1676.707737
8,AUD,1.671630
9,AWG,2.066660


## 5. Error handling

API requests can fail for many reasons: the server is down, the URL is wrong, or your
internet connection is lost. It is good practice to check the status code before
processing the response.

In [10]:
# Example: requesting a URL that does not exist
response = requests.get("https://api.open-meteo.com/v1/nonexistent")
print("Status code:", response.status_code)

if response.status_code == 200:
    print("Success")
else:
    print("Request failed")

Status code: 404
Request failed


Common HTTP status codes:

| Code | Meaning |
|---|---|
| 200 | OK - the request was successful |
| 400 | Bad Request - something was wrong with your request |
| 401 | Unauthorised - you need an API key |
| 404 | Not Found - the URL does not exist |
| 429 | Too Many Requests - you have exceeded the rate limit |

## 6. A note on API keys

Many APIs require an API key for authentication. When using API keys:

1. **Never hardcode them** in your notebook or script.
2. **Never upload them** to GitHub (add config files to `.gitignore`).
3. Store them in a separate file (e.g. `config.json`) or as environment variables. This will be practiced during this week's exercise session.

Example pattern for loading a key from a file:

```python
import json

with open("config.json") as f:
    config = json.load(f)

api_key = config["api_key"]
```